# Feature Analysis & Preprocessing Rationale

This notebook investigates which textual features carry sentiment signal in Twitter data,
and uses those findings to justify our preprocessing decisions.

**Sections:**
1. Dataset overview & class distribution
2. Social feature analysis (URLs, @mentions, #hashtags, emojis)
3. Sentiment-aware stopword preservation
4. Dual preprocessing: why transformers need different cleaning
5. VADER lexicon as orthogonal signal
6. Summary of preprocessing decisions

In [ ]:
import sys, os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

# Allow imports from project root
sys.path.insert(0, os.path.abspath(".."))

SEED = 42
np.random.seed(SEED)

## 1. Dataset Overview

In [ ]:
df = pd.read_csv("../dataset/train.csv", encoding="cp1252")
df = df[["textID", "text", "sentiment"]].dropna(subset=["text", "sentiment"])
df = df[df["text"].str.strip() != ""].reset_index(drop=True)

print(f"Total tweets: {len(df)}")
print(f"\nClass distribution:")
print(df["sentiment"].value_counts())
print(f"\nClass proportions:")
print(df["sentiment"].value_counts(normalize=True).round(3))

## 2. Social Feature Analysis

Twitter text contains several non-word features: URLs, @mentions, #hashtags, and emojis.
Before deciding to strip or keep them, we test whether they carry sentiment signal.

In [ ]:
# Detect social features
df["has_url"] = df["text"].str.contains(r"https?://\S+|www\.\S+", regex=True, na=False)
df["has_mention"] = df["text"].str.contains(r"@\w+", regex=True, na=False)
df["has_hashtag"] = df["text"].str.contains(r"#\w+", regex=True, na=False)

# Emoji detection (Unicode emoji ranges)
emoji_pattern = re.compile(
    "[" 
    "\U0001F600-\U0001F64F"  # emoticons
    "\U0001F300-\U0001F5FF"  # symbols & pictographs
    "\U0001F680-\U0001F6FF"  # transport & map
    "\U0001F1E0-\U0001F1FF"  # flags
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
    "]+"
)
df["has_emoji"] = df["text"].apply(lambda t: bool(emoji_pattern.search(str(t))))

# Summary table
features = ["has_url", "has_mention", "has_hashtag", "has_emoji"]
summary = []
for feat in features:
    count = df[feat].sum()
    pct = count / len(df) * 100
    summary.append({"feature": feat, "count": count, "pct": f"{pct:.1f}%"})

print(pd.DataFrame(summary).to_string(index=False))

In [ ]:
# Chi-squared test: is the presence of each feature independent of sentiment?
print("Chi-squared tests (H0: feature presence is independent of sentiment)")
print("=" * 70)

chi2_results = []
for feat in features:
    ct = pd.crosstab(df[feat], df["sentiment"])
    chi2, p, dof, expected = chi2_contingency(ct)
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    chi2_results.append({
        "feature": feat,
        "chi2": f"{chi2:.2f}",
        "p-value": f"{p:.2e}",
        "significant": sig,
    })
    print(f"{feat:15s}  chi2={chi2:8.2f}  p={p:.2e}  {sig}")

print("\n*** p<0.001, ** p<0.01, * p<0.05, ns = not significant")

### Interpretation

**URLs** are the strongest social signal. Tweets with URLs skew toward neutral sentiment —
they tend to be link-sharing rather than opinion-expressing. The chi-squared statistic is
very high and the association is statistically significant.

**@mentions** and **#hashtags** are sparse (<2% of tweets) but may still carry weak signal.
We preserve them in `clean_text` (keeping the `@` and `#` characters) so models can
learn from their presence if useful.

**Emojis** are very rare in this dataset. When present, they tend toward positive sentiment,
but the sample is too small to rely on.

### Preprocessing decision

- **Remove URL content** in classical cleaning: the URL string itself (http://...) is
  uninformative as a token — the sentiment signal is in the *presence* of a URL, not its
  text. For classical ML, URL strings create noisy high-dimensional features.
- **Keep @, # symbols**: cheap signal for classical models.
- For transformers (`light_clean`): preserve everything. The pretrained model can
  learn what to attend to.

In [ ]:
# Visualize: sentiment distribution for tweets WITH vs WITHOUT URLs
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (has, title) in zip(axes, [(True, "Tweets WITH URLs"), (False, "Tweets WITHOUT URLs")]):
    subset = df[df["has_url"] == has]
    counts = subset["sentiment"].value_counts(normalize=True)
    counts.reindex(["negative", "neutral", "positive"]).plot.bar(ax=ax, color=["#e74c3c", "#95a5a6", "#2ecc71"])
    ax.set_title(f"{title} (n={len(subset)})")
    ax.set_ylabel("Proportion")
    ax.set_ylim(0, 0.6)
    ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

## 3. Sentiment-Aware Stopword Preservation

Standard NLP practice removes stopwords to reduce noise. But for sentiment analysis,
some "stopwords" are the signal:

- **Negation**: "not happy" vs "happy" — flips polarity entirely
- **Intensifiers**: "very bad" vs "bad" — degree matters for ordinal sentiment
- **Contrast**: "good but expensive" — BUT signals sentiment reversal

We preserve 29 words from the NLTK English stopword list (179 total), leaving 150
words to filter.

In [ ]:
from preprocessing.preprocessing import SENTIMENT_KEEP, STOP_WORDS, _BASE_STOPWORDS

print(f"NLTK English stopwords: {len(_BASE_STOPWORDS)}")
print(f"Words preserved for sentiment: {len(SENTIMENT_KEEP)}")
print(f"Effective stopwords removed: {len(STOP_WORDS)}")

print(f"\nPreserved words by category:")
negation = sorted({"not", "no", "nor", "don", "aren", "isn", "wasn", "weren",
                    "hasn", "haven", "hadn", "couldn", "wouldn", "shouldn",
                    "didn", "won", "doesn", "mustn", "needn"})
intensifiers = sorted({"very", "too", "most", "more", "just", "only"})
contrast = sorted({"but", "however"})
other = sorted({"against", "few", "own"})

print(f"  Negation ({len(negation)}):     {negation}")
print(f"  Intensifiers ({len(intensifiers)}): {intensifiers}")
print(f"  Contrast ({len(contrast)}):     {contrast}")
print(f"  Other ({len(other)}):        {other}")

In [ ]:
# Demonstrate the impact: how often do preserved words appear in each sentiment class?
text_lower = df["text"].str.lower()

# Single regex pass over the corpus instead of 29 separate scans
keep_pattern = re.compile(r"\b(" + "|".join(re.escape(w) for w in sorted(SENTIMENT_KEEP)) + r")\b")
matches = text_lower.str.extractall(keep_pattern)  # (row_idx, match_num) -> word

rows = []
for word in sorted(SENTIMENT_KEEP):
    matched_rows = matches[matches[0] == word].index.get_level_values(0).unique()
    if len(matched_rows) < 10:
        continue
    ct = df.loc[matched_rows, "sentiment"].value_counts()
    total = len(matched_rows)
    rows.append({
        "word": word,
        "total": total,
        "neg_pct": ct.get("negative", 0) / total * 100,
        "neu_pct": ct.get("neutral", 0) / total * 100,
        "pos_pct": ct.get("positive", 0) / total * 100,
    })

keep_df = pd.DataFrame(rows).sort_values("total", ascending=False)
print("Sentiment distribution for preserved stopwords (top 15 by frequency):")
print(keep_df.head(15).to_string(index=False, float_format="{:.1f}".format))

### Key observation

Words like "not", "don", "no" skew negative — they are direct sentiment indicators.
Removing them as stopwords destroys information that classical models need.

**Empirical result (from sentiment_analysis experiments):**
BoW + Logistic Regression F1 improved from **0.6938 → 0.6997** (+0.006) solely from
this stopword fix — the single largest preprocessing improvement for classical models.

## 4. Dual Preprocessing: Why Transformers Need Different Cleaning

Classical ML models (BoW, TF-IDF) work on bag-of-tokens — they benefit from aggressive
cleaning that reduces vocabulary size and removes noise.

Transformer models (BERT, RoBERTa, sentence-transformers) are pretrained on natural
language with full grammar, punctuation, and function words. Stripping these produces
input that is *out-of-distribution* for the pretrained model.

**Concrete failure case:** When aggressive `clean_text` was fed to sentence-transformers
for clustering, it produced degenerate clusters — e.g., a cluster whose representative
tweets were just "know", "guess". The sentence transformer couldn't produce meaningful
embeddings from stripped-down text. Switching to `light_clean` (lowercase + whitespace
only) fixed this entirely.

In [ ]:
from preprocessing.preprocessing import clean_text, light_clean

# Side-by-side comparison
examples = [
    "I can't believe how NOT helpful this was... terrible!",
    "@user It's very good but too expensive #overpriced",
    "Not bad at all, actually quite impressed https://t.co/abc",
    "Don't you just LOVE Mondays?? #sarcasm",
]

print(f"{'Original':<55} {'clean_text':<40} {'light_clean'}")
print("=" * 130)
for ex in examples:
    ct = clean_text(ex)
    lt = light_clean(ex)
    print(f"{ex:<55} {ct:<40} {lt}")

Notice how `clean_text` preserves negation words ("not", "don") but strips punctuation
and non-alpha characters. `light_clean` preserves everything except case and extra whitespace.

### When to use which

| Model type | Preprocessing | Why |
|---|---|---|
| BoW, TF-IDF, Word2Vec | `clean_text` | Reduce vocabulary, keep sentiment tokens |
| Logistic Regression, SVM, etc. | `clean_text` | Same — these consume BoW/TF-IDF features |
| BERT, RoBERTa fine-tuning | `light_clean` or raw | Pretrained on natural language |
| Sentence-transformers (for clustering) | `light_clean` | Needs grammar for good embeddings |
| VADER | raw text | Rule-based; needs punctuation and caps |

## 5. VADER: Orthogonal Lexicon Signal

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based sentiment
analyzer tuned for social media text. It handles:
- Emoticons and slang
- Capitalization as emphasis ("GREAT" > "great")
- Degree modifiers ("very", "kind of")
- Conjunctions ("but" clauses)

VADER produces 4 scores: `compound` (overall -1 to +1), `pos`, `neg`, `neu`.

**Why include VADER?** It captures patterns that n-gram models miss — lexicon knowledge
is orthogonal to bag-of-words features. Adding VADER scores to TF-IDF+SVM improved
F1 by **+0.015** (0.6875 → 0.7022) in prior experiments.

In [ ]:
import nltk
nltk.download("vader_lexicon", quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

# Score all tweets
vader_scores = df["text"].apply(lambda t: vader.polarity_scores(str(t)))
vader_df = pd.DataFrame(vader_scores.tolist())
df["vader_compound"] = vader_df["compound"]
df["vader_pos"] = vader_df["pos"]
df["vader_neg"] = vader_df["neg"]
df["vader_neu"] = vader_df["neu"]

print(f"VADER scores computed for {len(df)} tweets")
print(df[["vader_compound", "vader_pos", "vader_neg", "vader_neu"]].describe().round(3))

In [ ]:
# VADER compound score distribution by sentiment class
fig, ax = plt.subplots(figsize=(10, 5))
for label, color in [("negative", "#e74c3c"), ("neutral", "#95a5a6"), ("positive", "#2ecc71")]:
    subset = df[df["sentiment"] == label]["vader_compound"]
    ax.hist(subset, bins=50, alpha=0.5, label=label, color=color, density=True)

ax.set_xlabel("VADER Compound Score")
ax.set_ylabel("Density")
ax.set_title("VADER Compound Score Distribution by Sentiment Class")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# VADER's own classification vs gold labels
def vader_classify(compound):
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"

df["vader_pred"] = df["vader_compound"].apply(vader_classify)

agreement = (df["vader_pred"] == df["sentiment"]).mean()
disagreement = 1 - agreement

print(f"VADER agrees with gold labels: {agreement:.1%}")
print(f"VADER disagrees with gold labels: {disagreement:.1%}")

# Confusion: VADER prediction vs gold
from sklearn.metrics import classification_report
print("\nVADER standalone classification report:")
print(classification_report(df["sentiment"], df["vader_pred"], digits=4))

### VADER discussion

VADER alone achieves roughly **F1 ~ 0.45-0.50 macro** as a zero-learning baseline.
This is useful in two ways:

1. **As additional features**: VADER's 4 scores (compound, pos, neg, neu) can be
   appended to TF-IDF feature vectors, giving classical models access to lexicon
   knowledge they can't learn from a small corpus alone.

2. **As a difficulty indicator**: tweets where VADER strongly disagrees with gold labels
   often involve sarcasm, implicit sentiment, or potential label noise — useful for
   error analysis.

VADER does **not** replace learned models — it complements them.

## 6. Summary of Preprocessing Decisions

| Decision | Rationale | Evidence |
|---|---|---|
| Two preprocessing functions | Transformers need natural language; classical ML needs sparse tokens | Degenerate clusters from aggressive cleaning |
| Preserve 29 sentiment stopwords | Negation/intensifiers carry sentiment signal | +0.006 F1 for BoW+LR |
| Keep @/# in clean_text | Weak but free signal | chi2 tests |
| Remove URL text in clean_text | URL strings are noisy tokens; presence (not content) carries signal | chi2=57.38 for URL presence |
| Light clean for transformers | Lowercase + whitespace only preserves pretrained knowledge | Fixed degenerate clustering |
| VADER as supplementary features | Orthogonal lexicon signal | +0.015 F1 for TF-IDF+SVM |

### Usage

```python
import sys; sys.path.insert(0, '..')
from preprocessing.preprocessing import preprocess_df

df = preprocess_df(df, text_col='text')
# df now has 'clean_text' and 'light_text' columns
```